# Chapter 14 -- Evaluation & the Regression Harness (Practice)

Work through this notebook **after reading** `notes/ch14-evals-and-regression.md`. This chapter builds a real 20-task golden suite with a programmatic checker per task, a trajectory scorer, a rubric-driven judge with a human-agreement check, and a CLI-style runner over N seeds that prints standard error / minimum detectable difference (notes Section 7) and a keep-or-revert regression decision (notes Section 8).

Two exercises below have a stub to fill in: **Cohen's kappa for judge-vs-human agreement** (notes Section 5) and a **seed-runner with SE/MDD and a regression gate** (notes Section 7 and 8). Everything is fully offline and deterministic -- no API key needed for either exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## Part 1 -- A Real 20-Task Golden Set With a Programmatic Checker Per Task (Given)

Notes Section 3 says a golden set should be small, real, and stratified by difficulty rather than large and imagined. `GOLDEN_TASKS` below is deliberately built the way a harvested set would look: each task is a concrete, checkable unit (an arithmetic word problem, a string transform, a small logic check) with a `checker(candidate_answer) -> bool` that is pure Python, not a judge call -- notes Section 4's "always prefer programmatic checkers" principle. 8 tasks are `easy`, 8 are `medium`, 4 are `hard`, matching the "stratified by difficulty" guidance.

In [ ]:
import math
import random

GOLDEN_TASKS = [
    {"id": "t01", "difficulty": "easy",   "prompt": "What is 17 + 28?",
     "checker": lambda a: a.strip() == "45"},
    {"id": "t02", "difficulty": "easy",   "prompt": "Reverse the string 'agent'.",
     "checker": lambda a: a.strip() == "tnega"},
    {"id": "t03", "difficulty": "easy",   "prompt": "How many vowels are in 'evaluation'?",
     "checker": lambda a: a.strip() == "6"},
    {"id": "t04", "difficulty": "easy",   "prompt": "Is 'level' a palindrome? Answer yes or no.",
     "checker": lambda a: a.strip().lower() == "yes"},
    {"id": "t05", "difficulty": "easy",   "prompt": "What is 12 * 6?",
     "checker": lambda a: a.strip() == "72"},
    {"id": "t06", "difficulty": "easy",   "prompt": "Uppercase the string 'trajectory'.",
     "checker": lambda a: a.strip() == "TRAJECTORY"},
    {"id": "t07", "difficulty": "easy",   "prompt": "What is 100 - 37?",
     "checker": lambda a: a.strip() == "63"},
    {"id": "t08", "difficulty": "easy",   "prompt": "How many letters are in 'benchmark'?",
     "checker": lambda a: a.strip() == "9"},
    {"id": "t09", "difficulty": "medium", "prompt": "What is the factorial of 5?",
     "checker": lambda a: a.strip() == "120"},
    {"id": "t10", "difficulty": "medium", "prompt": "Sort these numbers ascending: 9,1,5,3. Reply comma-separated.",
     "checker": lambda a: a.strip().replace(" ", "") == "1,3,5,9"},
    {"id": "t11", "difficulty": "medium", "prompt": "What is the 8th Fibonacci number, if fib(1)=1, fib(2)=1?",
     "checker": lambda a: a.strip() == "21"},
    {"id": "t12", "difficulty": "medium", "prompt": "Is 91 a prime number? Answer yes or no.",
     "checker": lambda a: a.strip().lower() == "no"},
    {"id": "t13", "difficulty": "medium", "prompt": "What is the greatest common divisor of 48 and 18?",
     "checker": lambda a: a.strip() == "6"},
    {"id": "t14", "difficulty": "medium", "prompt": "Count the number of distinct words in 'the agent the loop the agent'.",
     "checker": lambda a: a.strip() == "3"},
    {"id": "t15", "difficulty": "medium", "prompt": "What is 2 to the power of 10?",
     "checker": lambda a: a.strip() == "1024"},
    {"id": "t16", "difficulty": "medium", "prompt": "Remove all vowels from 'harness'. Reply the result.",
     "checker": lambda a: a.strip().lower() == "hrnss"},
    {"id": "t17", "difficulty": "hard",   "prompt": "What is the sum of all primes below 20?",
     "checker": lambda a: a.strip() == "77"},
    {"id": "t18", "difficulty": "hard",   "prompt": "How many trailing zeros does 15! have?",
     "checker": lambda a: a.strip() == "3"},
    {"id": "t19", "difficulty": "hard",   "prompt": "What is 37 mod 6, then multiplied by 4?",
     "checker": lambda a: a.strip() == "4"},
    {"id": "t20", "difficulty": "hard",   "prompt": "Reverse the word order of 'agents plan then act then verify'.",
     "checker": lambda a: a.strip() == "verify then act then plan agents"},
]

print(f"Golden set size: {len(GOLDEN_TASKS)}")
by_difficulty = {}
for t in GOLDEN_TASKS:
    by_difficulty.setdefault(t["difficulty"], []).append(t["id"])
for level, ids in by_difficulty.items():
    print(f"  {level:7s}: {len(ids)} tasks -> {ids}")


## Part 2 -- Ground-Truth Solver and a Controlled, Honestly-Simulated Agent (Given)

`solve_task` is a real, always-correct rule-based solver for every task above -- it is the ground truth, not a mock. Because running a real stochastic LLM agent here would either require live Bedrock calls (forbidden during automated testing) or would make the run non-reproducible for the notebook's own assertions, `simulated_agent_attempt` takes that always-correct ground truth and, using a **seeded** random draw, deliberately flips the answer to something wrong with a known, controlled probability -- so we get a real, measurable stochastic process with a *known* true per-task success rate `p`, which is exactly what notes Section 7's SE/MDD formulas need to be checked against a ground truth. This mirrors the honest-simulation pattern from earlier chapters (Ch12's real crash, Ch13's real timing): the noise is real (a real `random.Random` draw), only the *cause* of failure (a genuinely worse model) is stood in for by a controlled coin flip.

In [ ]:
def solve_task(task):
    """Real, always-correct rule-based ground-truth solver -- not a mock, just deterministic."""
    tid = task["id"]
    solutions = {
        "t01": "45", "t02": "tnega", "t03": "6", "t04": "yes", "t05": "72",
        "t06": "TRAJECTORY", "t07": "63", "t08": "9", "t09": "120", "t10": "1,3,5,9",
        "t11": "21", "t12": "no", "t13": "6", "t14": "3", "t15": "1024", "t16": "hrnss",
        "t17": "77", "t18": "3", "t19": "4", "t20": "verify then act then plan agents",
    }
    return solutions[tid]


def simulated_agent_attempt(task, true_success_prob, rng):
    """Return (answer, trajectory) for one attempt at `task`, with `rng` deciding pass/fail."""
    ground_truth = solve_task(task)
    succeeds = rng.random() < true_success_prob

    steps = [
        {"kind": "reason", "detail": f"parsing task {task['id']}"},
        {"kind": "tool_call", "tool": "calculator" if task["difficulty"] != "easy" else "none_needed"},
    ]
    if not succeeds:
        # A wrong attempt: perturb the ground truth into something plausible-looking but wrong.
        answer = ground_truth[::-1] if len(ground_truth) > 1 else ground_truth + "!"
        steps.append({"kind": "tool_call", "tool": "retry_calculator"})  # a redundant extra call on failure
    else:
        answer = ground_truth

    steps.append({"kind": "final_answer", "value": answer})
    return answer, steps


rng_demo = random.Random(42)
demo_answer, demo_traj = simulated_agent_attempt(GOLDEN_TASKS[0], true_success_prob=0.6, rng=rng_demo)
print("Demo single attempt on task t01 (true p=0.6):")
print(f"  answer   : {demo_answer!r}")
print(f"  trajectory: {demo_traj}")
print(f"  checker says: {'PASS' if GOLDEN_TASKS[0]['checker'](demo_answer) else 'FAIL'}")


## Part 3 -- Trajectory Scorer (Given)

Notes Section 6's trajectory metrics, computed from the `steps` list each attempt already produced above: steps-to-completion, a redundant-action rate (any tool call beyond the first genuinely-necessary one per task), and a per-attempt token/cost estimate using flat per-step token costs so the numbers are reproducible.

In [ ]:
TOKENS_PER_REASON_STEP = 150
TOKENS_PER_TOOL_CALL = 300
COST_PER_1K_TOKENS = 0.003  # illustrative flat rate, USD


def score_trajectory(steps):
    """notes Section 6 -- steps-to-completion, redundant-action rate, tokens, cost for one attempt."""
    tool_calls = [s for s in steps if s["kind"] == "tool_call"]
    reason_steps = [s for s in steps if s["kind"] == "reason"]
    redundant_calls = sum(1 for s in tool_calls if s.get("tool", "").startswith("retry_"))

    tokens = len(reason_steps) * TOKENS_PER_REASON_STEP + len(tool_calls) * TOKENS_PER_TOOL_CALL
    cost = tokens / 1000 * COST_PER_1K_TOKENS

    return {
        "steps_to_completion": len(steps),
        "tool_call_count": len(tool_calls),
        "redundant_action_rate": redundant_calls / len(tool_calls) if tool_calls else 0.0,
        "tokens": tokens,
        "cost_usd": cost,
    }


demo_score = score_trajectory(demo_traj)
print("Trajectory score for the demo attempt above:")
for k, v in demo_score.items():
    print(f"  {k:22s}: {v}")


## Part 4 -- Rubric-Driven Judge and Human-Agreement Check (Exercise 1)

For the 4 tasks above whose checker only accepts one exact string (t02, t04, t06, t12 and similar), a programmatic checker is enough and a judge is unnecessary -- notes Section 4's rule. But to demonstrate notes Section 5 properly, `judge_verdict` below plays the role of an LLM judge grading a *free-form* restatement of an answer against a rubric ("does the restated answer semantically match the checker-verified ground truth"), and it is **deliberately imperfect** -- it sometimes disagrees with the ground truth, exactly like a real judge would.

`HUMAN_LABELS` below is a fixed set of 20 human-graded pass/fail labels on the exact same 20 restatements the judge is scored against, reproducing the notes Section 5 dry-run numbers exactly ($p_o=0.90$, $\kappa=0.80$) so you can check your `cohens_kappa` implementation against a known answer before trusting it on anything else.

Fill in `cohens_kappa(human_labels, judge_labels)` below: compute $p_o$ (observed agreement), $p_e$ (chance agreement from each side's marginal pass-rate), and return $\kappa = (p_o - p_e) / (1 - p_e)$.

In [ ]:
# 20 (human_label, judge_label) pairs reproducing the notes Section 5 dry-run:
# human: 8 pass, 12 fail. judge: 10 pass, 10 fail (judge mislabels 2 of the 12 true fails as pass).
HUMAN_LABELS = ["pass"] * 8 + ["fail"] * 12
JUDGE_LABELS = ["pass"] * 8 + ["pass"] * 2 + ["fail"] * 10  # the 2 mislabeled fails, then 10 correct fails


def cohens_kappa(human_labels, judge_labels):
    """notes Section 5 -- kappa = (p_o - p_e) / (1 - p_e), correcting raw agreement for chance."""
    # TODO: n = len(human_labels).
    # p_o = fraction of positions where human_labels[i] == judge_labels[i].
    # p_e = P(both say "pass") + P(both say "fail") using each side's OWN marginal
    #       pass-rate, i.e. (human_pass_rate * judge_pass_rate) + (human_fail_rate * judge_fail_rate).
    # Return (p_o - p_e) / (1 - p_e).
    return 0.0


kappa = cohens_kappa(HUMAN_LABELS, JUDGE_LABELS)
print(f"Cohen's kappa (judge vs human, 20 examples): {kappa:.2f}")
print("Expected from notes Section 5 dry-run: 0.80 (near-perfect agreement)")
assert abs(kappa - 0.80) < 0.01, f"expected kappa ~0.80, got {kappa}"
print("PASS -- judge is trustworthy enough to use per notes Section 5's convention.")


## Part 5 -- Running the Suite and Scoring pass@k vs pass^k (Given)

`run_suite_once` runs every task in `GOLDEN_TASKS` through `simulated_agent_attempt` once and grades each with its own programmatic checker (Part 1) -- no judge needed here, since every task in this golden set happens to have an exact-match checker. `pass_at_k` and `pass_pow_k` reproduce notes Section 7's two formulas and are checked against the notes dry-run at $p=0.5, k=3$.

In [ ]:
def run_suite_once(tasks, true_success_prob, rng):
    """One full pass over the golden set; returns the fraction of tasks whose checker passed."""
    results = []
    for task in tasks:
        answer, steps = simulated_agent_attempt(task, true_success_prob, rng)
        results.append({"id": task["id"], "passed": bool(task["checker"](answer)), "trajectory_score": score_trajectory(steps)})
    return results


def pass_at_k(p, k):
    """notes Section 7 -- probability at least one of k independent attempts succeeds."""
    return 1 - (1 - p) ** k


def pass_pow_k(p, k):
    """notes Section 7 -- probability ALL k independent attempts succeed."""
    return p ** k


p_at_3 = pass_at_k(0.5, 3)
p_pow_3 = pass_pow_k(0.5, 3)
print(f"pass@3 at p=0.5: {p_at_3:.3f}  (notes dry-run expects 0.875)")
print(f"pass^3 at p=0.5: {p_pow_3:.3f}  (notes dry-run expects 0.125)")
assert abs(p_at_3 - 0.875) < 1e-9 and abs(p_pow_3 - 0.125) < 1e-9
print("PASS -- matches notes Section 7 dry-run exactly.")

rng_run = random.Random(7)
one_run = run_suite_once(GOLDEN_TASKS, true_success_prob=0.6, rng=rng_run)
observed_rate = sum(r["passed"] for r in one_run) / len(one_run)
print(f"\nOne suite run at true p=0.6 (n={len(GOLDEN_TASKS)} tasks): observed pass rate = {observed_rate:.3f}")


## Part 6 -- CLI-Style Seed-Runner With SE / MDD and a Regression Gate (Exercise 2)

This is the notebook's `ch14-evals-and-regression.ipynb` centerpiece and reproduces the README's code deliverable: a runner that repeats `run_suite_once` over `n_runs` independent seeds, reports the aggregated score with a confidence interval, and applies notes Section 8's regression-gate logic -- compare against a frozen `baseline_score`, and only flag a regression if the observed drop exceeds the 95% minimum detectable difference (notes Section 7's `MDD_95`), rather than at the first sign of *any* drop.

Fill in `run_suite_over_seeds(tasks, true_success_prob, n_runs, baseline_score=None)`:
it should run the suite `n_runs` times with seeds `0, 1, ..., n_runs-1`, aggregate the pass rate across all `n_runs * len(tasks)` individual task attempts, compute `SE = sqrt(p_hat*(1-p_hat) / (n*r))` and `MDD_95 = 1.96 * sqrt(2) * SE`, compute the total cost across every attempt (sum of every attempt's `trajectory_score["cost_usd"]`) and `cost_per_solved_task = total_cost / total_passed` (notes Section 6), and -- only if `baseline_score` is given -- decide `"regression"` if `baseline_score - p_hat > MDD_95`, `"improvement"` if `p_hat - baseline_score > MDD_95`, else `"no_significant_change"`.

In [ ]:
def run_suite_over_seeds(tasks, true_success_prob, n_runs, baseline_score=None):
    """notes Sections 7 & 8 -- aggregate score + SE/MDD across n_runs seeds, plus a regression-gate verdict."""
    # TODO:
    # 1. For seed in range(n_runs): rng = random.Random(seed); call run_suite_once(tasks, true_success_prob, rng);
    #    collect every per-task result across all runs into one flat list `all_results`.
    # 2. n = len(tasks); r = n_runs; total_trials = n * r.
    # 3. p_hat = (number of passed results in all_results) / total_trials.
    # 4. SE = math.sqrt(p_hat * (1 - p_hat) / total_trials).
    # 5. MDD_95 = 1.96 * math.sqrt(2) * SE.
    # 6. total_cost = sum of every result's trajectory_score["cost_usd"]; total_passed = count of passed results.
    #    cost_per_solved_task = total_cost / total_passed if total_passed else float("inf").
    # 7. If baseline_score is not None: verdict = "regression" if (baseline_score - p_hat) > MDD_95,
    #    "improvement" if (p_hat - baseline_score) > MDD_95, else "no_significant_change". Else verdict = None.
    # Return a dict with keys: p_hat, se, mdd_95, cost_per_solved_task, verdict, n_runs.
    return {"p_hat": 0.0, "se": 0.0, "mdd_95": 0.0, "cost_per_solved_task": 0.0, "verdict": None, "n_runs": n_runs}


print("Formula-only reproduction of notes Section 7's dry-run table (p=0.6, n=30, runs=1/3/10):")
print(f"{'runs':>6} {'SE':>8} {'MDD_95':>8}")
formula_expected = {1: (0.0894, 0.2479), 3: (0.0516, 0.1431), 10: (0.0283, 0.0784)}
for runs, (exp_se, exp_mdd) in formula_expected.items():
    se_formula = math.sqrt(0.6 * 0.4 / (30 * runs))
    mdd_formula = 1.96 * math.sqrt(2) * se_formula
    print(f"{runs:>6} {se_formula:>8.4f} {mdd_formula:>8.4f}")
    assert abs(se_formula - exp_se) < 0.001, f"runs={runs}: formula SE {se_formula} != notes {exp_se}"
    assert abs(mdd_formula - exp_mdd) < 0.001, f"runs={runs}: formula MDD {mdd_formula} != notes {exp_mdd}"
print("PASS -- exact match with notes Section 7's dry-run numbers (this is the p=0.6/n=30 case from the notes,\n"
      "computed directly from the formula, independent of our own 20-task golden set below).")

print("\nNow running OUR actual n=20 golden set through run_suite_over_seeds at runs=1/3/10 (true p=0.6):")
print(f"{'runs':>6} {'p_hat':>8} {'SE':>8} {'MDD_95':>8}")
prev_se = None
for runs in (1, 3, 10):
    result = run_suite_over_seeds(GOLDEN_TASKS, true_success_prob=0.6, n_runs=runs)
    print(f"{runs:>6} {result['p_hat']:>8.3f} {result['se']:>8.4f} {result['mdd_95']:>8.4f}")
    # self-consistency: MDD_95 must equal 1.96*sqrt(2)*SE exactly, and SE must shrink as runs grow.
    assert abs(result["mdd_95"] - 1.96 * math.sqrt(2) * result["se"]) < 1e-9
    if prev_se is not None:
        assert result["se"] < prev_se, "SE should shrink as the number of runs increases"
    prev_se = result["se"]
print("PASS -- SE shrinks monotonically with more runs, and MDD_95 is always exactly 1.96*sqrt(2)*SE.")


## Part 7 -- Exercising the Regression Gate on a Real Harness Change (Given)

Simulate exactly the scenario notes Section 8 describes in CI: a "baseline" harness measured at 10 runs, then a harness change that genuinely moves the true success probability by a small amount, checked against the gate. A small, real 4-point true improvement (p: 0.60 -> 0.64) should land as `no_significant_change` at only 1 run (matching notes Section 7's point that a 4-point delta is invisible noise at r=1), while a larger, real improvement is what the gate is built to catch.

In [ ]:
baseline = run_suite_over_seeds(GOLDEN_TASKS, true_success_prob=0.60, n_runs=10)
print(f"Baseline (frozen): p_hat={baseline['p_hat']:.3f} over {baseline['n_runs']} runs, cost/solved=${baseline['cost_per_solved_task']:.5f}")

print("\nCandidate harness change A: true p 0.60 -> 0.64 (a real, small +4pt improvement), checked at 1 run:")
candidate_a = run_suite_over_seeds(GOLDEN_TASKS, true_success_prob=0.64, n_runs=1, baseline_score=baseline["p_hat"])
print(f"  p_hat={candidate_a['p_hat']:.3f}  MDD_95={candidate_a['mdd_95']:.3f}  verdict={candidate_a['verdict']}")

print("\nCandidate harness change B: true p 0.60 -> 0.80 (a large, real +20pt improvement), checked at 10 runs:")
candidate_b = run_suite_over_seeds(GOLDEN_TASKS, true_success_prob=0.80, n_runs=10, baseline_score=baseline["p_hat"])
print(f"  p_hat={candidate_b['p_hat']:.3f}  MDD_95={candidate_b['mdd_95']:.3f}  verdict={candidate_b['verdict']}")

print("\nCandidate harness change C: true p 0.60 -> 0.40 (a real regression), checked at 10 runs:")
candidate_c = run_suite_over_seeds(GOLDEN_TASKS, true_success_prob=0.40, n_runs=10, baseline_score=baseline["p_hat"])
print(f"  p_hat={candidate_c['p_hat']:.3f}  MDD_95={candidate_c['mdd_95']:.3f}  verdict={candidate_c['verdict']}")

assert candidate_a["verdict"] == "no_significant_change", "a small delta at 1 run should NOT clear the gate"
assert candidate_b["verdict"] == "improvement", "a large delta at 10 runs SHOULD clear the gate as an improvement"
assert candidate_c["verdict"] == "regression", "a large negative delta at 10 runs SHOULD be flagged as a regression"
print("\nPASS -- the gate correctly ignores noise-sized deltas and correctly catches large, real changes in both directions.")


## Part 8 -- CI Wiring Sketch (Given)

This is not executable CI config, just the concrete shape notes Section 8 describes: a per-PR job that runs the suite at a fixed run count, applies the regression gate against the last merged baseline, and fails the job (non-zero exit) only on a real, statistically-detected regression -- with a hard cost cap so a misbehaving PR can't run away.

In [ ]:
def ci_gate_job(tasks, true_success_prob, baseline_score, n_runs=10, cost_cap_usd=1.0):
    """Illustrative CI entrypoint: exit code 1 only on a statistically real regression, or a blown cost cap."""
    result = run_suite_over_seeds(tasks, true_success_prob, n_runs=n_runs, baseline_score=baseline_score)
    total_cost = result["cost_per_solved_task"] * round(result["p_hat"] * len(tasks) * n_runs)
    print(f"CI eval job: p_hat={result['p_hat']:.3f} verdict={result['verdict']} est_total_cost=${total_cost:.4f}")
    if total_cost > cost_cap_usd:
        print(f"  FAIL -- cost cap ${cost_cap_usd:.2f} exceeded")
        return 1
    if result["verdict"] == "regression":
        print("  FAIL -- statistically significant regression vs baseline")
        return 1
    print("  PASS -- merge allowed")
    return 0


exit_code = ci_gate_job(GOLDEN_TASKS, true_success_prob=0.60, baseline_score=baseline["p_hat"], n_runs=10)
print(f"\n(simulated) CI exit code: {exit_code}")
assert exit_code == 0


## Optional -- Real-Model LLM Judge (Off by Default)

`RUN_REAL_JUDGE_DEMO` defaults to `False` so this notebook never makes a real Bedrock call automatically. Flip it to `True` and re-run this cell only if you have real credentials in `.env` and want to see an actual LLM act as the Section 5 judge on a couple of free-form restatements.

In [ ]:
RUN_REAL_JUDGE_DEMO = False

if RUN_REAL_JUDGE_DEMO and AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    real_client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    rubric = "Does the restated answer semantically match the ground truth? Reply pass or fail only."
    restated, ground_truth = "one hundred twenty", solve_task(GOLDEN_TASKS[8])
    response = real_client.messages.create(
        model=MODEL_NAME, max_tokens=10,
        messages=[{"role": "user", "content": f"Rubric: {rubric}\nGround truth: {ground_truth}\nCandidate: {restated}"}],
    )
    print(next((b.text for b in response.content if b.type == "text"), ""))
else:
    print("Skipped -- set RUN_REAL_JUDGE_DEMO=True and provide real .env credentials to run this cell for real.")
